In [1]:
from src.requirements import *
from src.ssl_large import *
from src.asr_model import *
from src.audio_handler import *
from src.tokenizer import *

In [2]:
data_path = os.path.join("data", "metadata_normal.tsv")
token_path = os.path.join("data", "tokenizer.json")
cache_path = os.path.join("data", "cache_mmap", "asr")
text_path = os.path.join("data", "text")

tokenizer = Tokenizer.load(token_path)
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 4
learning_rate = 1e-4

update_ver = 100_000

In [3]:
if not os.path.exists(token_path):
    text = load_text(text_path)
    tokenizer = Tokenizer()
    tokenizer.build_vocab(text)
    tokenizer.save(token_path)
else:
    tokenizer = Tokenizer.load(token_path)

vocab_size = len(tokenizer)
vocab = tokenizer.get_vocab()
print("Vocab size:", vocab_size)

Vocab size: 119


In [4]:
ssl_model = LargeSSLModel().to(device)
asr_model = ASRModel(
    ssl_model=ssl_model,
    vocab_size=len(tokenizer),
    hidden_dim=256,
    num_layers=4,
    dropout=0.2
).to(device)

c:\Users\Samir\miniconda3\envs\ml\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [5]:
checkpoint_dict = torch.load(os.path.join('models', 'ssl_model', f'ssl_model_checkpoint_{update_ver}.pth'))
ssl_state_dict = checkpoint_dict['model_state_dict']
ssl_model.load_state_dict(ssl_state_dict, strict=True)

<All keys matched successfully>

In [6]:

asr_dataset = ASRDataset(
    metadata_path=data_path,
    tokenizer=tokenizer,
    cache_dir=cache_path,
    top_db=TOP_DB
)

asr_dl = DataLoader(
    dataset=asr_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate_padding_asr
)

Loading metadata from cache...
  Cache ready! 157905 samples
  Total audio size: 21.89 GB
Encoding transcripts...


Encoding: 100%|██████████| 157905/157905 [00:03<00:00, 50631.96it/s]


In [7]:
dataset_size = len(asr_dataset)
accum = 8
epochs = 5
steps_per_epoch = dataset_size // (batch_size * accum)
T_max = epochs * steps_per_epoch
warmup = int(0.05 * T_max)
print("Dataset size: ", dataset_size)
print("Batch size: ", batch_size)
print("Steps per epoch: ", steps_per_epoch)
print("Tmax: ", T_max)
print("Warmup steps: ", warmup)

Dataset size:  157905
Batch size:  4
Steps per epoch:  4934
Tmax:  24670
Warmup steps:  1233


In [8]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, asr_model.parameters()),
    lr=3e-4,
    weight_decay=0.01
)
# Check optimizer has params
print(f"\nOptimizer managing {sum(p.numel() for group in optimizer.param_groups for p in group['params']):,} parameters")

loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-4,
    total_steps=20_000,
    pct_start=0.1,
    anneal_strategy='cos'
)
scaler = torch.GradScaler(device)


Optimizer managing 11,862,519 parameters


In [9]:
beam_decoder = torchaudio.models.decoder.ctc_decoder(
    lexicon=None,
    tokens=vocab,
    blank_token='<blank>',
    sil_token='।',
    unk_word=None,
    nbest=1,
    beam_size=50,
)

In [10]:
def load_asr_checkpoint(asr_model, checkpoint_path, device='cuda'):
    print(f"Loading ASR checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Load model state
    asr_model.load_state_dict(checkpoint['model_state_dict'])
    asr_model.to(device)
    
    # Load optimizer and scheduler if available
    optimizer = None
    scheduler = None
    epoch = 0
    
    if 'optimizer_state_dict' in checkpoint:
        print("✓ Optimizer state found in checkpoint")
        optimizer = checkpoint.get('optimizer_state_dict')
    
    if 'scheduler_state_dict' in checkpoint:
        print("✓ Scheduler state found in checkpoint")
        scheduler = checkpoint.get('scheduler_state_dict')
    
    if 'epoch' in checkpoint:
        epoch = checkpoint['epoch']
        print(f"✓ Resuming from epoch {epoch}")
    
    return asr_model, optimizer, scheduler, epoch


def save_asr_checkpoint(asr_model, optimizer, scheduler, epoch, save_path, 
                       best_wer=None, best_cer=None):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': asr_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
    }
    
    if best_wer is not None:
        checkpoint['best_wer'] = best_wer
    if best_cer is not None:
        checkpoint['best_cer'] = best_cer
    
    torch.save(checkpoint, save_path)
    print(f"✓ Checkpoint saved: {save_path}")

In [ ]:
def train_asr(asr_model, asr_dl, optimizer, scaler, scheduler, loss_fn, epochs, device):
    max_updates = 20_000
    num_updates = 0
    
    for epoch in range(epochs):
        print(f"Epoch [{epoch+1}/{epochs}]")
        total_loss = 0.0

        asr_model.train()
        
        for i, batch in enumerate(tqdm(asr_dl)):
            waveforms, targets, input_lengths, target_lengths = batch
            waveforms = waveforms.to(device)
            input_lengths = input_lengths.to(device)
            target_lengths = target_lengths.to(device)
            
            with torch.autocast(device_type=device, dtype=torch.float16):
                log_probs = asr_model(waveforms, input_lengths) / accum
                flat_targets = torch.cat(targets).to(device)
                loss = loss_fn(log_probs, flat_targets, input_lengths, target_lengths)
            
            scaler.scale(loss).backward()
            total_loss += loss.item() * accum
            
            if (i+1) % accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(asr_model.parameters(), max_norm=2.0)
                scaler.step(optimizer)
                scaler.update()
                
                scheduler.step()
                optimizer.zero_grad()
                num_updates += 1
            
            if num_updates % 1_000 == 0:
                save_path = os.path.join('models', 'asr_model', f'asr_model_prototype_{num_updates}.pth')
                save_checkpoint(asr_model, optimizer, scheduler, num_updates, save_path)
            
            if num_updates >= max_updates:
                break
        
        torch.cuda.empty_cache()
        avg_loss = total_loss / len(asr_dl)
        print(f"Epoch {epoch+1} - Avg Loss: {avg_loss:.4f}")
        
        if num_updates >= max_updates:
            print(f"Reached max updates ({max_updates})")
            break
    
    return asr_model

In [ ]:
# asr_model = train_asr(asr_model, asr_dl, asr_optimizer, scaler, scheduler, loss_fn, epochs, device)